# Dataset 3 Ratio Ablation
Builds **dataset3b–3e** (60/40 → 90/10 answerable/unanswerable) from:
- `dataset3.jsonl` — the existing 50/50 split (5k answerable + 5k unanswerable)
- `dataset1.jsonl` — source of extra answerable questions

All files are read from and written to **Google Drive**.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configure paths
Set `DATA_DIR` to wherever your `.jsonl` files live in Drive.

In [ ]:
from pathlib import Path

# ✏️  Change this to match your Drive folder
DATA_DIR = Path('/content/drive/MyDrive/abstention-data/data')

D1_PATH = DATA_DIR / 'dataset1.jsonl'
D3_PATH = DATA_DIR / 'dataset3.jsonl'   # existing 50/50
OUT_DIR = DATA_DIR / 'dataset3_ablation'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Sanity check
for p in [D1_PATH, D3_PATH]:
    status = '✓' if p.exists() else '✗  NOT FOUND'
    print(f'{status}  {p}')

## 3. Load dataset3 (50/50 baseline)
Split into answerable and unanswerable pools.

In [ ]:
import json
from collections import Counter

NO_ANSWER = '<NO-ANSWER>'

d3_answerable   = []
d3_unanswerable = []

with D3_PATH.open('r', encoding='utf-8') as f:
    for line in f:
        ex = json.loads(line)
        if ex.get('output') == NO_ANSWER:
            d3_unanswerable.append(ex)
        else:
            d3_answerable.append(ex)

print(f'dataset3  total        : {len(d3_answerable) + len(d3_unanswerable):,}')
print(f'  answerable           : {len(d3_answerable):,}')
print(f'  unanswerable         : {len(d3_unanswerable):,}')
print()
print('Unanswerable scenario breakdown:')
for scenario, count in Counter(
    ex.get('unanswerable_type', 'Unknown') for ex in d3_unanswerable
).most_common():
    print(f'  {scenario}: {count:,}')

## 4. Load extra answerable questions from dataset1
Exclude any IDs already present in dataset3 to avoid overlap.

In [ ]:
# IDs already in dataset3
d3_ids = {ex['id'] for ex in d3_answerable + d3_unanswerable}

extra_answerable = []

with D1_PATH.open('r', encoding='utf-8') as f:
    for line in f:
        ex = json.loads(line)
        if ex['id'] in d3_ids:
            continue
        # Normalise to the same field names used in dataset3
        extra_answerable.append({
            'id'    : ex['id'],
            'input' : ex.get('input', ex.get('question', '')).strip(),
            'output': ex.get('output', ex.get('answer', '')),
            'split' : ex.get('split', 'train'),
        })

print(f'Extra answerable from dataset1 (no overlap with dataset3): {len(extra_answerable):,}')
print()

# Max answerable we can ever use
full_answerable_pool = d3_answerable + extra_answerable
print(f'Full answerable pool (d3 + extra): {len(full_answerable_pool):,}')

## 5. Shuffle pools (fixed seed for reproducibility)

In [ ]:
import random

SEED = 42
random.seed(SEED)

# Shuffle once — every prefix is a random representative sample,
# so the 50/50 answerable slice is a strict subset of the 60/40 slice, etc.
random.shuffle(full_answerable_pool)
random.shuffle(d3_unanswerable)

print('Pools shuffled with seed', SEED)

## 6. Helper — proportional unanswerable subsampling
As the unanswerable count shrinks we sample proportionally from each scenario
type to keep the scenario mix balanced.

In [ ]:
from collections import defaultdict

def subsample_unanswerable(pool, n, seed=SEED):
    """Return n items from pool, preserving scenario type proportions."""
    rng = random.Random(seed)
    by_scenario = defaultdict(list)
    for ex in pool:
        by_scenario[ex.get('unanswerable_type', 'Unknown')].append(ex)

    total    = len(pool)
    selected = []
    remainder = n
    scenarios = sorted(by_scenario.keys())

    for i, scenario in enumerate(scenarios):
        bucket = by_scenario[scenario]
        quota  = remainder if i == len(scenarios) - 1 else round(n * len(bucket) / total)
        quota  = min(quota, len(bucket))
        selected.extend(rng.sample(bucket, quota))
        remainder -= quota

    rng.shuffle(selected)
    return selected

print('subsample_unanswerable() defined')

## 7. Generate dataset3b – 3e
| Name      | Ratio | Answerable | Unanswerable |
|-----------|-------|------------|-------------|
| dataset3b | 60/40 | 6,000      | 4,000       |
| dataset3c | 70/30 | 7,000      | 3,000       |
| dataset3d | 80/20 | 8,000      | 2,000       |
| dataset3e | 90/10 | 9,000      | 1,000       |

In [ ]:
TOTAL = 10_000

ABLATION_CONFIGS = [
    # (name,        pct_answerable)
    ('dataset3b',   60),
    ('dataset3c',   70),
    ('dataset3d',   80),
    ('dataset3e',   90),
]

summary = []

for name, pct_ans in ABLATION_CONFIGS:
    pct_unans = 100 - pct_ans
    n_ans     = int(TOTAL * pct_ans   / 100)
    n_unans   = int(TOTAL * pct_unans / 100)

    # Validate pool sizes
    if n_ans > len(full_answerable_pool):
        print(f'[ERROR] Not enough answerable examples for {name} '
              f'(need {n_ans:,}, have {len(full_answerable_pool):,})')
        continue
    if n_unans > len(d3_unanswerable):
        print(f'[ERROR] Not enough unanswerable examples for {name} '
              f'(need {n_unans:,}, have {len(d3_unanswerable):,})')
        continue

    ans_sample   = full_answerable_pool[:n_ans]       # prefix of shuffled pool
    unans_sample = subsample_unanswerable(d3_unanswerable, n_unans)

    dataset = ans_sample + unans_sample
    random.shuffle(dataset)

    out_path = OUT_DIR / f'{name}.jsonl'
    with out_path.open('w', encoding='utf-8') as fh:
        for ex in dataset:
            fh.write(json.dumps(ex, ensure_ascii=False) + '\n')

    unans_counts = Counter(
        ex.get('unanswerable_type', 'Unknown') for ex in unans_sample
    )
    summary.append({
        'name'         : name,
        'ratio'        : f'{pct_ans}/{pct_unans}',
        'n_answerable' : n_ans,
        'n_unanswerable': n_unans,
        'breakdown'    : dict(unans_counts),
        'path'         : str(out_path),
    })
    print(f'✓  {name}  ({pct_ans}/{pct_unans})  →  '
          f'{n_ans:,} answerable + {n_unans:,} unanswerable  →  {out_path.name}')

print(f'\nAll files written to: {OUT_DIR}')

## 8. Summary

In [ ]:
print(f"{'Name':<12} {'Ratio':<8} {'Answerable':>12} {'Unanswerable':>14}  Scenario breakdown")
print('-' * 95)

# Include the original dataset3 (50/50) as reference
baseline_counts = Counter(
    ex.get('unanswerable_type', 'Unknown') for ex in d3_unanswerable
)
ref_breakdown = ', '.join(f'{k}: {v}' for k, v in sorted(baseline_counts.items()))
print(f"{'dataset3 (ref)':<12} {'50/50':<8} {len(d3_answerable):>12,} {len(d3_unanswerable):>14,}  {ref_breakdown}")

for row in summary:
    breakdown = ', '.join(f'{k}: {v}' for k, v in sorted(row['breakdown'].items()))
    print(f"{row['name']:<12} {row['ratio']:<8} {row['n_answerable']:>12,} {row['n_unanswerable']:>14,}  {breakdown}")